<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_61_gpt2_to_llama3_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🦙 **Module 61 — GPT-2 → Llama 3 Architecture Conversion** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 8 · LLMs from Scratch (Raschka bonus)


# Module 61 — GPT-2 → Llama 3

> M56 built **GPT-2 124M**. Every modern LLM — Llama, Qwen, Mistral, DeepSeek, Gemma — is *almost* the same architecture, with four well-known swaps. This module performs each swap in code, on top of the M56 model, and ends with **Llama 3.2-1B weights loaded into your converted stack** producing real text.

### The four swaps

| Component | GPT-2 (2019) | Llama-style (2023→) |
|---|---|---|
| Normalisation | `LayerNorm` | **RMSNorm** |
| Positional encoding | learned absolute `pos_emb` | **RoPE** (rotary on Q/K) |
| MLP activation | `GELU(x·W₁)·W₂` | **SwiGLU** — `(silu(xW₁) ⊙ xW₃)·W₂` |
| Attention layout | MHA (one KV per Q head) | **GQA** — many Q heads share one KV head |

Each swap is a contained change. After all four you have a Llama block. After loading real weights you have Llama itself.

### What you'll cover
1. The four swaps in one paragraph each
2. **RMSNorm** — drop the mean, scale only
3. **RoPE** — rotate Q/K by angle proportional to position
4. **SwiGLU** — gated FFN with three weight matrices
5. **GQA** — grouped-query attention + the KV-cache savings
6. A complete **`LlamaBlock`** (replaces M56's `TransformerBlock`)
7. The full **`LlamaModel`** spec
8. Download Llama-3.2-1B from Hugging Face + load weights
9. Generate text from the converted stack
10. The bigger picture — what *else* changes (vocab, MoE, MLA)


## 1 · The four swaps

- **RMSNorm** — Pre-Norm 2.0. Drops the mean centring; just RMS-divide and scale. Trains as well, half the math, fewer FLOPs.
- **RoPE** — instead of *adding* a positional vector to embeddings, **rotate** the query and key vectors by an angle proportional to position. The dot product `q_i · k_j` then implicitly encodes `i − j`. Composable; works with KV cache; extrapolates better.
- **SwiGLU** — replace the GELU two-matrix MLP with a **gated** three-matrix MLP. Empirically nicer training curves; standard since PaLM.
- **GQA (Grouped-Query Attention)** — many query heads share a single KV head. KV cache shrinks `n_heads / n_kv_heads`× — the *single* biggest reason a Llama-3 8B can serve 32K-token contexts on one GPU.


## 2 · RMSNorm

In [ ]:
!pip -q install torch tiktoken huggingface_hub safetensors

In [ ]:
import torch, torch.nn as nn

class RMSNorm(nn.Module):
    """Llama's RMSNorm — no mean centring, no bias. Only RMS-divide + a learnable scale."""
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))   # gamma; no beta

    def forward(self, x):
        # rsqrt(mean(x^2) + eps)  →  x / sqrt(mean(x^2) + eps)
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x * rms)

**Why this works.** The mean term in LayerNorm exists to centre activations. Empirically, transformers don't need centring at the LN step — the residual stream handles drift on its own. RMSNorm is one fewer matrix operation per token per layer × `2 × n_layers` LNs per block = ~5-10% wall-clock win.

## 3 · RoPE — rotary positional embeddings

RoPE rotates each pair of dimensions in the Q/K vector by an angle `θ_d · m` where `m` is the position and `θ_d` is a per-dim base frequency.

$$\theta_d = 10000^{-2d/D}$$

For position `m`, dimension pair `(2d, 2d+1)`:

$$\begin{pmatrix}q'_{2d} \\ q'_{2d+1}\end{pmatrix} = \begin{pmatrix}\cos(m\theta_d) & -\sin(m\theta_d) \\ \sin(m\theta_d) & \cos(m\theta_d)\end{pmatrix} \begin{pmatrix}q_{2d} \\ q_{2d+1}\end{pmatrix}$$

The dot product `q_m · k_n` then only depends on the **relative** offset `m − n`.

In [ ]:
def precompute_rope_freqs(head_dim, max_seq_len, base=10_000.0, device="cpu"):
    """Returns (cos, sin) tables of shape (max_seq_len, head_dim//2)."""
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)                    # (T, head_dim/2)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):
    """x: (B, H, T, head_dim).  cos,sin: (T, head_dim/2).  Returns rotated x."""
    # split each head_dim into two halves and rotate them as (real, imag) pairs
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    cos = cos[None, None, : x.shape[-2], :]            # broadcast to (1,1,T,head_dim/2)
    sin = sin[None, None, : x.shape[-2], :]
    return torch.cat([x1 * cos - x2 * sin,
                      x1 * sin + x2 * cos], dim=-1)

# tiny demo
cos, sin = precompute_rope_freqs(head_dim=8, max_seq_len=4)
q = torch.randn(1, 2, 4, 8)                            # (B=1, H=2, T=4, head_dim=8)
print("rotated q[0,0,0,:]:", apply_rope(q, cos, sin)[0, 0, 0])

**Two practical notes.**
- The `(cos, sin)` tables are **precomputed** once (or extended with the context length). They're not learned.
- Llama-3 uses `base=500_000` (extended from 10 000) which yields a longer effective context window without changing the model.

## 4 · SwiGLU — gated MLP

In [ ]:
class SwiGLU(nn.Module):
    """Llama-style gated MLP: (silu(x W_gate) ⊙ x W_up) → W_down"""
    def __init__(self, d_model, d_ff, bias=False):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=bias)
        self.w_up   = nn.Linear(d_model, d_ff, bias=bias)
        self.w_down = nn.Linear(d_ff,    d_model, bias=bias)

    def forward(self, x):
        return self.w_down(torch.nn.functional.silu(self.w_gate(x)) * self.w_up(x))

**The shape change.** GPT-2's FFN has **2** matrices: `(d_model → 4·d_model)` then `(4·d_model → d_model)`. SwiGLU has **3** matrices and chooses `d_ff` differently (usually `~2/3 × 4·d_model ≈ 2.66·d_model` so total params match GPT-2's 4× FFN). The extra parameter count buys the **gating** signal — `silu(...)` is the *gate* and the second matrix is the *content* — and that consistently improves training quality.

## 5 · GQA — Grouped-Query Attention

In [ ]:
class GQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, rope_base=500_000.0):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.h_q  = n_heads
        self.h_kv = n_kv_heads
        self.hd   = d_model // n_heads
        self.group = n_heads // n_kv_heads          # how many Q heads share one KV head

        self.W_q = nn.Linear(d_model, n_heads    * self.hd, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.hd, bias=False)   # <-- shrunk
        self.W_v = nn.Linear(d_model, n_kv_heads * self.hd, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        cos, sin = precompute_rope_freqs(self.hd, max_seq_len, base=rope_base)
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)
        self.register_buffer("mask", torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool())

    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_q(x).view(B, T, self.h_q,  self.hd).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.h_kv, self.hd).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.h_kv, self.hd).transpose(1, 2)

        # RoPE on Q and K only (NOT V)
        Q = apply_rope(Q, self.rope_cos, self.rope_sin)
        K = apply_rope(K, self.rope_cos, self.rope_sin)

        # repeat KV heads so they line up with Q heads
        K = K.repeat_interleave(self.group, dim=1)
        V = V.repeat_interleave(self.group, dim=1)

        attn = (Q @ K.transpose(-2, -1)) / (self.hd ** 0.5)
        attn = attn.masked_fill(self.mask[:T, :T], -torch.inf)
        attn = torch.softmax(attn, dim=-1)
        out  = (attn @ V).transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(out)

**What's new vs M56's `MultiHeadAttention`.**
1. `W_k` and `W_v` project to **`n_kv_heads × head_dim`**, *not* `n_heads × head_dim`. That's the KV cache shrink.
2. **RoPE is applied to Q and K**, not V. (V doesn't need positional information — the softmax already weighted its mixing.)
3. We `repeat_interleave` the KV heads so the shapes line up at the dot product.
4. No QKV bias (Llama dropped it).

For Llama-3-8B: `n_heads=32, n_kv_heads=8`. KV cache is **4× smaller** than a vanilla MHA model of the same size. That's why an 8B model holds 32K context on a single GPU.

## 6 · `LlamaBlock` — pre-norm + RMSNorm + GQA + SwiGLU

In [ ]:
class LlamaBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1   = RMSNorm(cfg["d"])
        self.attn = GQA(cfg["d"], cfg["n_heads"], cfg["n_kv_heads"], cfg["ctx"], cfg["rope_base"])
        self.n2   = RMSNorm(cfg["d"])
        self.mlp  = SwiGLU(cfg["d"], cfg["d_ff"])

    def forward(self, x):
        x = x + self.attn(self.n1(x))     # pre-norm residual (same shape as M56)
        x = x + self.mlp(self.n2(x))
        return x

**Look at the diff vs M56's `TransformerBlock`.** Same control flow, same pre-norm, same residual. Every change is *inside* a sub-module:

| | M56 | M61 |
|---|---|---|
| `n1`, `n2` | `LayerNorm` | `RMSNorm` |
| `attn` | `MultiHeadAttention` | `GQA` (with RoPE) |
| `mlp` | `GELU(W₁·x)·W₂` | `SwiGLU(x)` |

That's how reading a Llama paper feels less scary once you've built a GPT — it's the same skeleton, four limbs swapped.

## 7 · The `LlamaModel`

In [ ]:
class LlamaModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.embed_tokens = nn.Embedding(cfg["vocab"], cfg["d"])
        self.layers = nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg["n_layers"])])
        self.norm   = RMSNorm(cfg["d"])
        self.lm_head = nn.Linear(cfg["d"], cfg["vocab"], bias=False)
        # Llama-3 does NOT tie input/output embeddings (different from GPT-2)
    def forward(self, idx):
        x = self.embed_tokens(idx)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(self.norm(x))

# A Llama-3.2-1B sized config (16 layers · d_model 2048 · n_heads 32 · n_kv_heads 8)
CFG_LLAMA_1B = {
    "vocab": 128_256,        # Llama-3 BPE
    "ctx":   2048,            # demo cap — real model: 131K
    "d":     2048,
    "n_heads": 32,
    "n_kv_heads": 8,
    "n_layers": 16,
    "d_ff": 8192,
    "rope_base": 500_000.0,
}
torch.manual_seed(0)
# don't actually instantiate at 1B params in Colab unless you have a GPU — just inspect the spec
print(CFG_LLAMA_1B)

**Three more Llama-3 specifics** the spec captures:
- `vocab = 128_256` (vs GPT-2's 50 257). Llama uses a larger BPE so each token covers more characters on average — fewer tokens per document.
- `n_kv_heads = 8` while `n_heads = 32` → GQA with group size 4.
- Embeddings are **not tied** to the LM head (Llama trains both separately). That's a small Llama-vs-GPT-2 inconsistency to know about when loading weights.

## 8 · Load Llama-3.2-1B weights

Real models live on Hugging Face. Llama-3.2-1B-Instruct is gated — you must accept the licence once on the HF page, then `huggingface_hub` will let you download it.

```python
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
import json, pathlib

repo = "meta-llama/Llama-3.2-1B"
local = snapshot_download(repo, allow_patterns=["*.json","*.safetensors","tokenizer*"])
files = sorted(pathlib.Path(local).glob("*.safetensors"))
state = {}
for f in files:
    state.update(load_file(f))
print("keys (first 10):", list(state.keys())[:10])
```

The state-dict keys look like:
```
model.embed_tokens.weight             # token embedding
model.layers.0.input_layernorm.weight        # RMSNorm (gamma only)
model.layers.0.self_attn.q_proj.weight       # GQA — Q
model.layers.0.self_attn.k_proj.weight       # GQA — K (smaller)
model.layers.0.self_attn.v_proj.weight       # GQA — V (smaller)
model.layers.0.self_attn.o_proj.weight       # output projection
model.layers.0.post_attention_layernorm.weight   # RMSNorm
model.layers.0.mlp.gate_proj.weight          # SwiGLU gate
model.layers.0.mlp.up_proj.weight            # SwiGLU up
model.layers.0.mlp.down_proj.weight          # SwiGLU down
model.norm.weight                            # final RMSNorm
lm_head.weight                                # LM head (not tied)
```

Mapping into your model is then **mechanical** — there's no `c_attn` fusing this time:

In [ ]:
def load_llama_weights(model, state, cfg):
    model.embed_tokens.weight.data.copy_(state["model.embed_tokens.weight"])
    for i, layer in enumerate(model.layers):
        p = f"model.layers.{i}"
        layer.n1.weight.data.copy_(state[f"{p}.input_layernorm.weight"])
        layer.n2.weight.data.copy_(state[f"{p}.post_attention_layernorm.weight"])
        layer.attn.W_q.weight.data.copy_(state[f"{p}.self_attn.q_proj.weight"])
        layer.attn.W_k.weight.data.copy_(state[f"{p}.self_attn.k_proj.weight"])
        layer.attn.W_v.weight.data.copy_(state[f"{p}.self_attn.v_proj.weight"])
        layer.attn.W_o.weight.data.copy_(state[f"{p}.self_attn.o_proj.weight"])
        layer.mlp.w_gate.weight.data.copy_(state[f"{p}.mlp.gate_proj.weight"])
        layer.mlp.w_up.weight.data.copy_(state[f"{p}.mlp.up_proj.weight"])
        layer.mlp.w_down.weight.data.copy_(state[f"{p}.mlp.down_proj.weight"])
    model.norm.weight.data.copy_(state["model.norm.weight"])
    model.lm_head.weight.data.copy_(state["lm_head.weight"])
    return model
print("loader defined — call load_llama_weights(model, state, CFG_LLAMA_1B)")

**Notice what's *missing* compared to M56's loader.**
- No transpose. HF's state-dict is *already* PyTorch (out, in) shape — no TF re-shaping.
- No fused-matrix split. Q / K / V are stored separately.
- No re-tying of the LM head. Llama-3 trains it separately from `embed_tokens`.

That's why HF's official models are easier to host than re-implementing from a TF dump.

## 9 · Generate

In [ ]:
# (Conceptual — you'd use the AutoTokenizer that ships with the model)
# from transformers import AutoTokenizer
# tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
# ids = tok("The capital of France is", return_tensors="pt").input_ids
#
# # Reuse M57's generate() with EOS = tok.eos_token_id (128009 for Llama-3 instruct)
# out = generate(model, ids, max_new=20, context_size=CFG_LLAMA_1B["ctx"], temperature=0.7, top_k=50)
# print(tok.decode(out[0]))
print("With the cache from M60 this runs ~10× faster on the same hardware as the naive loop.")

**You now have a working Llama 3.2-1B.** The architecture you wrote, the weights you loaded, generating real text. Add the KV cache from M60 and you have a serviceable inference engine in pure PyTorch.

## 10 · The bigger picture — what else changes

Beyond the four swaps, the Llama family ecosystem has a few more flavours:

| Family | Extra trick |
|---|---|
| **Llama 3 / Qwen 2.5 / Mistral 7B** | GQA + SwiGLU + RoPE (what we built) |
| **Mixtral / Qwen-MoE / DeepSeek-MoE** | **MoE FFN** — replace the SwiGLU with `n_experts` SwiGLUs + a learned router; only `top_k` experts run per token. Compute is constant; parameter count balloons. (M24 covered MoE conceptually.) |
| **DeepSeek-V2 / V3** | **MLA** — Multi-head Latent Attention: project KV down to a tiny shared latent, expand back at attention time. 10× smaller KV cache. (See M24.) |
| **Mistral 7B** | **Sliding-window attention** with optional sinks. |
| **Gemma 2 / 3** | RMSNorm **inside** residuals (post-norm-ish), logit soft-capping. |
| **Olmo 2** | LayerNorm **on Q and K** before attention scores. |
| **Phi-3** | RMSNorm, dense MLP, larger d_ff:d_model ratio. |

Once you know the four swaps in this module, every modern open-weight LLM paper reads as: "we use the Llama recipe plus *one* of the tricks above." There is no more "from scratch" — the recipe is shared.

## ✅ Recap
- Four swaps turn GPT-2 into Llama: **LayerNorm → RMSNorm**, **abs pos-emb → RoPE**, **GELU MLP → SwiGLU**, **MHA → GQA**.
- Each swap is contained to one sub-module — `LlamaBlock` has the same skeleton as `TransformerBlock`.
- HF state-dicts map mechanically (no transpose, no fused split, no tying) — much easier than the TF GPT-2 dump.
- Modern families (Mixtral, DeepSeek, Gemma, Phi) layer a single extra trick on top of this recipe.
- Plug the cache from M60 in and you have a serviceable inference engine.

Next: **M62 — DPO from Scratch**.
